# Probability-of-Default Credit Scorecard — Walkthrough

An executable narrative of the end-to-end scorecard build. Each section calls the project's
modules in `../src` and shows live output. **All data is synthetic** (see `src/data_generation.py`).

**Pipeline:** Data → WOE/IV → Feature selection → Logistic scorecard → Validation →
Champion–challenger → Monitoring (PSI/CSI) → Lending strategy & P&L.

In [1]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
np.set_printoptions(suppress=True)

## 1 · Data & target

A consumer-lending book combining application, bureau and behavioural/alternative data, with a 90+ DPD target over a 12-month window.

In [2]:
from data_generation import generate_credit_data
dev, oot = generate_credit_data(n_dev=60_000, n_oot=20_000)
dev = dev.reset_index(drop=True); oot = oot.reset_index(drop=True)
print(f"Development: {dev.shape[0]:,} rows | bad rate {dev.default_flag.mean()*100:.2f}%")
print(f"Out-of-time: {oot.shape[0]:,} rows | bad rate {oot.default_flag.mean()*100:.2f}%")
dev.head()

Development: 60,000 rows | bad rate 11.54%
Out-of-time: 20,000 rows | bad rate 20.29%


,age,annual_income,employment_length_yrs,residential_status,employment_status,bureau_score,months_on_book,revolving_utilization,num_delinq_24m,num_inquiries_6m,num_open_trades,num_public_records,requested_loan_amt,dti_ratio,interest_rate,digital_engagement_score,default_flag,vintage
0,28,"24,900.0000",7.1000,RENT,CONTRACT,574,104,66.7000,1,4,11,0,"1,000.0000",33.9000,16.4200,63,1,2023H1_DEV
1,27,"44,600.0000",6.2000,MORTGAGE,SALARIED,615,152,47.6000,0,1,14,0,"17,800.0000",37.2000,13.6600,49,1,2023H1_DEV
2,34,"57,500.0000",12.6000,RENT,CONTRACT,669,100,51.5000,1,1,10,0,"12,400.0000",31.7000,12.8000,57,0,2023H1_DEV
3,45,"67,100.0000",10.6000,MORTGAGE,CONTRACT,802,118,4.9000,0,1,9,0,"27,500.0000",31.2000,6.3100,50,0,2023H1_DEV
4,21,"34,700.0000",0.0000,MORTGAGE,SALARIED,575,130,25.0000,0,1,8,0,"18,600.0000",29.4000,15.4100,57,0,2023H1_DEV


## 2 · Weight of Evidence & Information Value

Supervised, monotonic binning replaces each predictor with the WOE of its bin (`ln(%good/%bad)`); Information Value ranks predictive power.

In [3]:
from woe_iv import WOEEncoder
y = dev["default_flag"]; X = dev.drop(columns=["default_flag","vintage"])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

enc = WOEEncoder(max_bins=6, min_bin_frac=0.05, enforce_monotonic=True).fit(Xtr, ytr)
enc.iv_summary().round(4)

,IV,strength
bureau_score,0.6238,very_strong
interest_rate,0.5426,very_strong
revolving_utilization,0.4035,strong
num_delinq_24m,0.2955,medium
dti_ratio,0.2241,medium
num_inquiries_6m,0.2008,medium
age,0.1062,medium
digital_engagement_score,0.0743,weak
annual_income,0.0376,weak
employment_length_yrs,0.0242,weak


In [4]:
# Worked binning table: bureau score (note the monotone WOE)
enc.binning_table("bureau_score").round(4)

,bin,n,pct_total,n_good,n_bad,event_rate,woe,iv
0,"(-inf, 557.5]",2100,0.0500,1280,820,0.3905,-1.5906,0.2142
1,"(557.5, 582.5]",2611,0.0622,1955,656,0.2512,-0.9442,0.0781
2,"(582.5, 606.5]",4282,0.1020,3470,812,0.1896,-0.5837,0.0432
3,"(606.5, 628.5]",5260,0.1252,4511,749,0.1424,-0.2407,0.0080
4,"(628.5, 658.5]",8496,0.2023,7628,868,0.1022,0.1372,0.0036
5,"(658.5, inf]",19251,0.4584,18307,944,0.0490,0.9287,0.2768


## 3 · Feature selection

Four gates: IV band → drop endogenous (offered rate) → VIF < 5 → correct coefficient sign.

In [5]:
from scorecard import (Scorecard, select_features, prune_multicollinearity,
                       variance_inflation_factors)
cands = select_features(enc.iv_summary()["IV"], exclude=("interest_rate",))
woe_tr, woe_te = enc.transform(Xtr), enc.transform(Xte)
kept, dropped = prune_multicollinearity(woe_tr[[f+"_woe" for f in cands]], vif_threshold=5.0)
print("Excluded (endogenous): interest_rate")
print("Dropped (VIF):", dropped or "none")
print("Final features:", [c[:-4] for c in kept])

Excluded (endogenous): interest_rate
Dropped (VIF): none
Final features: ['bureau_score', 'revolving_utilization', 'num_delinq_24m', 'dti_ratio', 'num_inquiries_6m', 'age', 'digital_engagement_score', 'annual_income', 'employment_length_yrs']


## 4 · Scorecard development & scaling

Logistic regression on WOE, scaled to points (600 @ 50:1, PDO 20).

In [6]:
sc = Scorecard(pdo=20, base_score=600, base_odds=50).fit(woe_tr, ytr, kept)
wrong = sc.wrong_sign_features()
if wrong:
    kept = [c for c in kept if c not in wrong]
    sc = Scorecard(pdo=20, base_score=600, base_odds=50).fit(woe_tr, ytr, kept)
print("Wrong-sign features:", wrong or "none — all coefficients correctly signed")
pd.Series(sc.coef_, name="beta").round(4).to_frame()

Wrong-sign features: none — all coefficients correctly signed


,beta
bureau_score_woe,-0.5827
revolving_utilization_woe,-0.4467
num_delinq_24m_woe,-0.3915
dti_ratio_woe,-0.5415
num_inquiries_6m_woe,-0.3961
age_woe,-0.5529
digital_engagement_score_woe,-0.6315
annual_income_woe,-0.1649
employment_length_yrs_woe,-0.1547


In [7]:
pts = sc.build_points_table(enc)
pts[pts.feature == "bureau_score"]

,feature,bin,n,event_rate,woe,beta,points
0,bureau_score,"(-inf, 557.5]",2100,0.3905,-1.5906,-0.5827,34
1,bureau_score,"(557.5, 582.5]",2611,0.2512,-0.9442,-0.5827,45
2,bureau_score,"(582.5, 606.5]",4282,0.1896,-0.5837,-0.5827,51
3,bureau_score,"(606.5, 628.5]",5260,0.1424,-0.2407,-0.5827,57
4,bureau_score,"(628.5, 658.5]",8496,0.1022,0.1372,-0.5827,63
5,bureau_score,"(658.5, inf]",19251,0.0490,0.9287,-0.5827,76


## 5 · Validation

Discrimination (KS, Gini, AUC), rank-ordering and calibration — in-time and out-of-time.

In [8]:
import validation as val
woe_oot = enc.transform(oot.drop(columns=["default_flag","vintage"]))
yoot = oot["default_flag"]
for name, w, t in [("Train", woe_tr, ytr), ("Test", woe_te, yte), ("OOT", woe_oot, yoot)]:
    print(val.full_report(t, sc.predict_proba(w), sc.score(w), name))

{'segment': 'Train', 'n': 42000, 'bad_rate': 0.1155, 'ks': 0.3834, 'gini': 0.5197, 'auc': 0.7599, 'brier': 0.08987, 'log_loss': 0.30878}
{'segment': 'Test', 'n': 18000, 'bad_rate': 0.1154, 'ks': 0.3822, 'gini': 0.5157, 'auc': 0.7578, 'brier': 0.0901, 'log_loss': 0.30963}
{'segment': 'OOT', 'n': 20000, 'bad_rate': 0.2029, 'ks': 0.3936, 'gini': 0.5284, 'auc': 0.7642, 'brier': 0.13498, 'log_loss': 0.42885}


In [9]:
# Score-band rank-ordering table
band = val.score_band_table(yte, sc.score(woe_te), n_bands=10)
band

,band,n,min_score,max_score,bads,goods,bad_rate,cum_bad_pct,cum_good_pct,ks,lift
0,1,1800,442.1000,516.6000,675,1125,0.3750,0.3248,0.0707,0.2541,3.2500
1,2,1800,516.6000,532.4000,373,1427,0.2072,0.5043,0.1603,0.3440,1.7900
2,3,1800,532.4000,542.8000,275,1525,0.1528,0.6367,0.2561,0.3806,1.3200
3,4,1800,542.8000,551.9000,190,1610,0.1056,0.7281,0.3572,0.3709,0.9100
4,5,1800,551.9000,559.9000,156,1644,0.0867,0.8032,0.4604,0.3428,0.7500
5,6,1800,559.9000,566.8000,150,1650,0.0833,0.8754,0.5641,0.3113,0.7200
6,7,1800,566.8000,573.6000,99,1701,0.0550,0.9230,0.6709,0.2521,0.4800
7,8,1800,573.7000,580.7000,76,1724,0.0422,0.9596,0.7792,0.1804,0.3700
8,9,1800,580.7000,589.6000,43,1757,0.0239,0.9803,0.8895,0.0908,0.2100
9,10,1800,589.6000,618.6000,41,1759,0.0228,1.0000,1.0000,0.0000,0.2000


## 6 · Champion–challenger

Benchmark the interpretable model against XGBoost & Random Forest.

In [10]:
from challengers import fit_challengers
ch = fit_challengers(Xtr, ytr, Xte, yte, exclude=("interest_rate",))
champ = val.full_report(yte, sc.predict_proba(woe_te), sc.score(woe_te), "Logistic (champion)")
pd.DataFrame([champ] + ch["reports"]).set_index("segment")[["ks","gini","auc","brier"]].round(4)

,ks,gini,auc,brier
segment,,,,
Logistic (champion),0.3822,0.5157,0.7578,0.0901
XGBoost,0.3923,0.5201,0.7600,0.1895
RandomForest,0.3899,0.5213,0.7607,0.1791


> The challengers add **< 0.01 Gini** while being far less calibrated and not natively explainable — the interpretable champion wins for a regulated lending decision.

## 7 · Monitoring — PSI & CSI

Does the model stay healthy as the population shifts?

In [11]:
import monitoring as mon
psi, psi_tbl = mon.population_stability_index(sc.score(woe_tr), sc.score(woe_oot))
print(f"Score PSI (dev -> OOT) = {psi:.4f}  ->  investigate zone")
mon.characteristic_stability_index(enc, Xtr, oot.drop(columns=['default_flag','vintage']), kept)

Score PSI (dev -> OOT) = 0.2428  ->  investigate zone


,characteristic,csi,flag
0,revolving_utilization,0.3815,shifted
1,bureau_score,0.1346,moderate
2,dti_ratio,0.0852,stable
3,num_delinq_24m,0.0553,stable
4,num_inquiries_6m,0.0318,stable
5,digital_engagement_score,0.0037,stable
6,annual_income,0.0005,stable
7,employment_length_yrs,0.0003,stable
8,age,0.0002,stable


> Discrimination held out-of-time while PSI flagged red and the bad rate doubled — the model still *ranks* risk correctly but is *mis-calibrated*. Action: **recalibrate, not rebuild**.

## 8 · Lending strategy & P&L

Turn the score into an approval policy and quantify it.

In [12]:
import business_impact as biz
s_te = sc.score(woe_te)
profit_df, best = biz.profit_optimised_cutoff(yte, s_te, margin_per_good=1200, loss_per_bad=9000)
print("Profit-optimal cutoff:", best)
summary, verdict = biz.swap_set_analysis(yte, s_te, Xte["bureau_score"].values,
                                         new_cutoff=best["cutoff_score"], incumbent_cutoff=620)
summary

Profit-optimal cutoff: {'cutoff_score': 545.2, 'approval_rate': 0.6744, 'approved_bad_rate': 0.0577, 'expected_profit': 7426800.0, 'profit_per_app': 412.6}


,group,n,bads,bad_rate
0,Approved by both,10958,584,0.0533
1,Swap-in (scorecard only),1177,115,0.0977
2,Swap-out (incumbent only),1809,311,0.1719


In [13]:
print("Swap-set verdict:", verdict)

Swap-set verdict: {'swap_in_bad_rate': 0.0977, 'swap_out_bad_rate': 0.1719, 'scorecard_adds_value': True, 'new_approval_rate': 0.6742, 'incumbent_approval_rate': 0.7093}


---
### Summary

A transparent logistic scorecard delivering **KS ≈ 38 / Gini ≈ 0.52**, validated in- and out-of-time,
monitored with PSI/CSI, and translated into a profit-optimised lending strategy — the full lifecycle a
credit-risk model development lead owns, under SR 11-7 governance.

See [`outputs/Credit_Risk_Scorecard_Report.html`](../outputs/Credit_Risk_Scorecard_Report.html) for the
full visual report.